# Clustered conformal prediction: numbers for the Results section

The appendix documents the *mechanism* of clustered conformal prediction in this
design: how often the clustering step runs, how many classes reach the null
cluster, how often the prediction sets coincide with the marginal ones. What is
missing is any measure of its *performance*: neither WCU nor AvgSize for
clustered appears anywhere in the paper.

That gap leaves an assertion unsupported. The appendix states that clustered
conformal prediction shows slightly worse worst-class under-coverage than the
marginal procedure at some calibration sizes, and explains why — but the reader
is never shown the number. This notebook produces it.

**Why compare against the marginal procedure and not classwise.** The two
questions are different. Clustered against classwise asks which method is
better, and cannot be settled numerically because the two target different
coverage guarantees. Clustered against marginal asks whether splitting the
calibration sample bought anything: when clustering is skipped, the fallback
threshold is computed on the proper calibration subsample while the marginal
procedure uses the whole sample, so the comparison isolates the cost of the
split. If clustered is worse, the withheld sample was not compensated by any
pooling.

**Why AvgSize and not only WCU.** Worst-class under-coverage can always be
improved by being conservative. If clustered has both worse WCU and larger sets
it is dominated and can be described as such; if it has worse WCU but smaller
sets, it is a trade-off and must be presented as one. Without AvgSize there is
no way to know which case applies — and that distinction is what Referee 1's
seventh comment asks for.

**Why these three calibration sizes.** They correspond to the three regimes the
appendix table already identifies: at an expected rare-class count of 20 the
clustering step never runs, at 50 it runs in 87.3% of replications with about
half the classes still in the null cluster, at 100 it runs in every replication.
Reporting performance at these three points ties it to the mechanism rather than
producing three unconnected numbers. The first two also appear in the main
paired table, so the reader can compare without anything being repeated.

## Setup

In [ ]:
#CODE_DIR    = "/content"
#RESULTS_DIR = "/content/results_R300"
#OUT_DIR     = "/content/paper_outputs"

from google.colab import drive; drive.mount("/content/drive")
BASE        = "/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision"
CODE_DIR    = f"{BASE}/code"
RESULTS_DIR = f"{BASE}/results_R300"
OUT_DIR     = f"{BASE}/paper_outputs"

import sys, os
sys.path.insert(0, CODE_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np, pandas as pd
import metrics as M

pd.set_option("display.width", 200)

ALPHA, SCEN, MODEL = 0.10, "s3_nuisance", "XGBoost"
#LEVELS = [500, 1250, 2500]        # expected rare-class counts 20, 50, 100
LEVELS = [125, 250, 500, 1250, 2500, 5000]

df = M.load(RESULTS_DIR)
df = df[(df.scenario == SCEN) & (df.model == MODEL) & (df.alpha == ALPHA)]
rep  = M.per_replication(df)
summ = M.summarize(rep)
xs   = df.groupby("n_cal")["n_cal_expected_rarest"].first()
print(f"{len(df):,} long rows | {rep.rep.nunique()} replications")

Mounted at /content/drive
230,400 long rows | 300 replications


## 1. Performance of clustered conformal prediction

Three methods at three calibration sizes. Marginal APS is the comparison that
isolates the cost of the split; classwise orbit averaging is included at the
largest size only, since the main paired table already covers the other two.

In [ ]:
ROWS = [("clustered",          "Clustered APS"),
        ("marginal_plain",      "Marginal APS"),
        ("classwise_orbitavg",  "Classwise orbit-averaged APS")]

out = []
for n in LEVELS:
    for key, label in ROWS:
        r = summ[(~summ.train_aug) & (summ.method == key) & (summ.n_cal == n)]
        if r.empty:
            continue
        r = r.iloc[0]
        out.append({"E[N_rare]": int(round(xs[n])), "n_cal": n, "Method": label,
                    "WCU": r.WCU, "WCU MCSE": r.WCU_mcse,
                    "AvgSize": r.AvgSize, "AvgSize MCSE": r.AvgSize_mcse,
                    "MacroSize": r.MacroSize, "CovMarginal": r.CovMarginal})

perf = pd.DataFrame(out)
display(perf.round({"WCU": 4, "WCU MCSE": 4, "AvgSize": 3, "AvgSize MCSE": 3,
                    "MacroSize": 3, "CovMarginal": 3}))
perf.to_csv(f"{OUT_DIR}/clustered_performance.csv", index=False)

,E[N_rare],n_cal,Method,WCU,WCU MCSE,AvgSize,AvgSize MCSE,MacroSize,CovMarginal
0,5,125,Clustered APS,0.2179,0.0049,1.688,0.015,1.977,0.904
1,5,125,Marginal APS,0.2179,0.0049,1.688,0.015,1.977,0.904
2,5,125,Classwise orbit-averaged APS,0.0696,0.0041,4.876,0.041,4.933,0.930
3,10,250,Clustered APS,0.2351,0.0048,1.649,0.013,1.926,0.901
4,10,250,Marginal APS,0.2356,0.0048,1.647,0.012,1.922,0.901
5,10,250,Classwise orbit-averaged APS,0.0761,0.0039,2.786,0.035,3.121,0.915
6,20,500,Clustered APS,0.2319,0.0040,1.636,0.011,1.914,0.900
7,20,500,Marginal APS,0.2335,0.0040,1.630,0.010,1.906,0.899
8,20,500,Classwise orbit-averaged APS,0.0650,0.0026,2.116,0.018,2.499,0.905
9,50,1250,Clustered APS,0.2226,0.0041,1.606,0.010,1.869,0.881


## 2. Clustered against marginal, paired

The appendix asserts that clustered is worse than marginal at some calibration
sizes. Paired differencing settles it: the two arms share the data, the splits,
the fitted model and the randomisation, so the difference isolates the
procedure. A positive WCU difference means clustered under-covers the worst
class more than marginal does.

In [ ]:
pairs = []
for metric in ["WCU", "AvgSize"]:
    d = M.paired_delta(rep[~rep.train_aug], metric,
                       "marginal_plain", "clustered")
    d["metric"] = metric
    pairs.append(d)

paired = pd.concat(pairs, ignore_index=True)
paired = paired[paired.n_cal.isin(LEVELS)].copy()
paired["E[N_rare]"] = paired.n_cal.map(xs).round().astype(int)
paired["t"] = paired.mean_delta / paired.mcse

display(paired[["metric", "E[N_rare]", "mean_delta", "mcse", "t",
                "frac_negative", "n"]].round(4))

print("mean_delta = clustered minus marginal.")
print("  WCU     > 0  ->  clustered under-covers the worst class more")
print("  AvgSize > 0  ->  clustered produces larger sets")
paired.to_csv(f"{OUT_DIR}/clustered_vs_marginal_paired.csv", index=False)

,metric,E[N_rare],mean_delta,mcse,t,frac_negative,n
0,WCU,5,0.0000,0.0000,NaN,0.0000,300
1,WCU,10,-0.0006,0.0005,-1.2182,0.1100,300
2,WCU,20,-0.0017,0.0005,-3.0980,0.4400,300
3,WCU,50,-0.0045,0.0011,-4.1210,0.4567,300
4,WCU,100,-0.0876,0.0052,-16.9869,0.7933,300
5,WCU,200,-0.0013,0.0008,-1.6432,0.3100,300
6,AvgSize,5,0.0000,0.0000,NaN,0.0000,300
7,AvgSize,10,0.0026,0.0015,1.7678,0.0700,300
8,AvgSize,20,0.0058,0.0015,3.8338,0.2467,300
9,AvgSize,50,-0.0278,0.0050,-5.6092,0.6533,300


mean_delta = clustered minus marginal.
  WCU     > 0  ->  clustered under-covers the worst class more
  AvgSize > 0  ->  clustered produces larger sets


### Is clustered dominated?

Dominated means worse on both criteria. If instead it trades coverage for size,
the paragraph has to be written as a trade-off — which is what the referee's
request for pros and cons is about.

In [ ]:
for n in LEVELS:
    w = paired[(paired.metric == "WCU") & (paired.n_cal == n)]
    s = paired[(paired.metric == "AvgSize") & (paired.n_cal == n)]
    if w.empty or s.empty:
        continue
    w, s = w.iloc[0], s.iloc[0]

    # A difference counts as real when it exceeds two Monte Carlo standard
    # errors. Both directions matter: clustered can be better as well as worse,
    # and treating only positive differences as informative would report a
    # large improvement as "comparable".
    tol_w, tol_s = 2 * w.mcse, 2 * s.mcse
    cov  = ("worse"  if w.mean_delta >  tol_w else
            "better" if w.mean_delta < -tol_w else "comparable")
    size = ("larger" if s.mean_delta >  tol_s else
            "smaller" if s.mean_delta < -tol_s else "comparable")

    print(f"E[N_rare]={int(round(xs[n])):>4}  "
          f"dWCU={w.mean_delta:+.4f} ({w.mcse:.4f})  "
          f"dSize={s.mean_delta:+.3f} ({s.mcse:.3f})  ->  "
          f"coverage {cov}, sets {size}")

E[N_rare]=   5  dWCU=+0.0000 (0.0000)  dSize=+0.000 (0.000)  ->  coverage comparable, sets comparable
E[N_rare]=  10  dWCU=-0.0006 (0.0005)  dSize=+0.003 (0.001)  ->  coverage comparable, sets comparable
E[N_rare]=  20  dWCU=-0.0017 (0.0005)  dSize=+0.006 (0.002)  ->  coverage better, sets larger
E[N_rare]=  50  dWCU=-0.0045 (0.0011)  dSize=-0.028 (0.005)  ->  coverage better, sets smaller
E[N_rare]= 100  dWCU=-0.0876 (0.0052)  dSize=+0.252 (0.009)  ->  coverage better, sets larger
E[N_rare]= 200  dWCU=-0.0013 (0.0008)  dSize=+0.003 (0.002)  ->  coverage comparable, sets comparable


## 3. Mechanism alongside performance

The execution rate and the null-cluster count come from the appendix
diagnostics; putting them beside WCU and AvgSize shows whether the degradation
tracks the state of the clustering step rather than occurring at random.

In [ ]:
from ding_conformal_utils import get_clustering_parameters, get_quantile_threshold
from simulate import IMBALANCE_PROFILE

PI, K = IMBALANCE_PROFILE / IMBALANCE_PROFILE.sum(), len(IMBALANCE_PROFILE)

def clustering_diagnostic(n_cal, alpha, R=300, seed=20260808):
    rng, thresh = np.random.default_rng(seed), get_quantile_threshold(alpha)
    rows = []
    for _ in range(R):
        y = rng.choice(K, size=n_cal, p=PI)
        cts = np.bincount(y, minlength=K)
        n_min = max(cts.min(), thresh)
        n_clust, Mc = get_clustering_parameters(int((cts >= n_min).sum()), n_min)
        frac = n_clust / n_min if n_min else 0.0
        n_rare = int((np.bincount(y[rng.random(n_cal) < frac],
                                  minlength=K) < thresh).sum())
        rows.append((Mc, frac, n_rare, (K - n_rare > Mc) and (Mc > 1)))
    d = pd.DataFrame(rows, columns=["M", "gamma", "n_null", "runs"])
    return {"n_cal": n_cal, "M": int(d.M.median()),
            "gamma": round(d.gamma.median(), 3),
            "P_executed": round(d.runs.mean(), 3),
            "null_classes": round(d.n_null.mean(), 2)}

mech = pd.DataFrame([clustering_diagnostic(n, ALPHA) for n in LEVELS])
coll = M.clustered_diagnostic(rep[~rep.train_aug])
coll = coll[coll.n_cal.isin(LEVELS)][["n_cal", "frac_collapsed"]]

joined = (perf[perf.Method == "Clustered APS"]
            .merge(mech, on="n_cal").merge(coll, on="n_cal"))
display(joined[["E[N_rare]", "M", "gamma", "P_executed", "null_classes",
                "frac_collapsed", "WCU", "AvgSize"]].round(3))
joined.to_csv(f"{OUT_DIR}/clustered_mechanism_and_performance.csv", index=False)

,E[N_rare],M,gamma,P_executed,null_classes,frac_collapsed,WCU,AvgSize
0,5,0,0.000,0.000,8.00,1.000,0.218,1.688
1,10,0,0.000,0.000,7.89,0.810,0.235,1.649
2,20,0,0.062,0.000,6.84,0.243,0.232,1.636
3,50,2,0.087,0.873,3.96,0.013,0.223,1.606
4,100,4,0.092,1.000,0.74,0.000,0.146,1.864
5,200,9,0.094,0.013,0.01,0.100,0.226,1.632


## 3b. Marginal coverage of clustered conformal prediction

Cluster-conditional coverage does not imply marginal coverage when the partition
is incomplete: Proposition 1 of Ding et al. guarantees coverage conditionally on
membership of one of the M clusters, and classes assigned to the null cluster
fall outside that statement. Marginal coverage is the mixture over all classes,
so it inherits no guarantee from the cluster-level one when a non-negligible
share of classes is null.

The prediction is therefore specific: any shortfall should appear where the
partition is partial, and not where clustering is skipped entirely or where
almost every class is clustered.

In [ ]:
# Marginal coverage across the grid, against the nominal level.
cov = (summ[(~summ.train_aug) & summ.method.isin(
            ["clustered", "marginal_plain", "classwise_orbitavg"])]
       [["n_cal", "method", "CovMarginal", "CovMarginal_mcse"]]
       .pivot(index="n_cal", columns="method",
              values=["CovMarginal", "CovMarginal_mcse"]))
display(cov.round(4))

print(f"nominal level: {1 - ALPHA:.2f}\n")
for n in LEVELS:
    r = summ[(~summ.train_aug) & (summ.method == "clustered")
             & (summ.n_cal == n)]
    if r.empty:
        continue
    r = r.iloc[0]
    z = (r.CovMarginal - (1 - ALPHA)) / r.CovMarginal_mcse
    print(f"E[N_rare]={int(round(xs[n])):>4}  clustered CovMarginal "
          f"{r.CovMarginal:.4f} (MCSE {r.CovMarginal_mcse:.4f})  "
          f"{z:+.1f} MCSE from nominal")

# Paired against the marginal procedure: sharper than comparing the two means.
d = M.paired_delta(rep[~rep.train_aug], "CovMarginal",
                   "marginal_plain", "clustered")
d = d[d.n_cal.isin(LEVELS)].copy()
d["E[N_rare]"] = d.n_cal.map(xs).round().astype(int)
d["t"] = d.mean_delta / d.mcse
print()
display(d[["E[N_rare]", "mean_delta", "mcse", "t", "frac_negative", "n"]].round(5))

CovMarginal                            CovMarginal_mcse                         
method classwise_orbitavg clustered marginal_plain classwise_orbitavg clustered marginal_plain
n_cal                                                                                         
125                0.9304    0.9045         0.9045             0.0013    0.0017         0.0017
250                0.9148    0.9011         0.9008             0.0011    0.0011         0.0011
500                0.9054    0.9000         0.8992             0.0009    0.0009         0.0008
1250               0.9023    0.8813         0.9006             0.0006    0.0009         0.0006
2500               0.9023    0.8965         0.9001             0.0005    0.0006         0.0005
5000               0.9004    0.9005         0.9004             0.0004    0.0005         0.0005

nominal level: 0.90

E[N_rare]=   5  clustered CovMarginal 0.9045 (MCSE 0.0017)  +2.7 MCSE from nominal
E[N_rare]=  10  clustered CovMarginal 0.9011 (MCSE 0.0011)  +1.0 MCSE from nominal
E[N_rare]=  20  clustered CovMarginal 0.9000 (MCSE 0.0009)  -0.0 MCSE from nominal
E[N_rare]=  50  clustered CovMarginal 0.8813 (MCSE 0.0009)  -20.0 MCSE from nominal
E[N_rare]= 100  clustered CovMarginal 0.8965 (MCSE 0.0006)  -5.7 MCSE from nominal
E[N_rare]= 200  clustered CovMarginal 0.9005 (MCSE 0.0005)  +1.1 MCSE from nominal



,E[N_rare],mean_delta,mcse,t,frac_negative,n
0,5,0.00000,0.00000,NaN,0.00000,300
1,10,0.00029,0.00023,1.27401,0.07000,300
2,20,0.00074,0.00025,2.98874,0.21000,300
3,50,-0.01938,0.00081,-23.93340,0.86667,300
4,100,-0.00367,0.00058,-6.36522,0.61000,300
5,200,0.00009,0.00010,0.84557,0.36333,300


## 4. Sentences for the Results paragraph

In [ ]:
def g(method, n, col):
    r = perf[(perf.Method == method) & (perf.n_cal == n)]
    return float(r.iloc[0][col]) if len(r) else float("nan")

print("=== Clustered conformal prediction ===")
for n in LEVELS:
    e = int(round(xs[n]))
    row = mech[mech.n_cal == n].iloc[0]
    print(f"\nE[N_rare] = {e}  (n_cal = {n})")
    print(f"  clustering executed in {100*row.P_executed:.1f}% of replications, "
          f"{row.null_classes:.2f} classes in the null cluster on average")
    print(f"  clustered : WCU {g('Clustered APS', n, 'WCU'):.3f}  "
          f"AvgSize {g('Clustered APS', n, 'AvgSize'):.3f}")
    print(f"  marginal  : WCU {g('Marginal APS', n, 'WCU'):.3f}  "
          f"AvgSize {g('Marginal APS', n, 'AvgSize'):.3f}")
    w = paired[(paired.metric == 'WCU') & (paired.n_cal == n)]
    if len(w):
        w = w.iloc[0]
        print(f"  paired difference (clustered - marginal): "
              f"{w.mean_delta:+.4f} (MCSE {w.mcse:.4f}, t = {w.t:.2f})")

n = LEVELS[-1]
print(f"\n=== At E[N_rare] = {int(round(xs[n]))}, where clustering always runs ===")
for m in ["Clustered APS", "Classwise orbit-averaged APS"]:
    print(f"  {m:32s} WCU {g(m, n, 'WCU'):.3f}  AvgSize {g(m, n, 'AvgSize'):.3f}")

=== Clustered conformal prediction ===

E[N_rare] = 5  (n_cal = 125)
  clustering executed in 0.0% of replications, 8.00 classes in the null cluster on average
  clustered : WCU 0.218  AvgSize 1.688
  marginal  : WCU 0.218  AvgSize 1.688
  paired difference (clustered - marginal): +0.0000 (MCSE 0.0000, t = nan)

E[N_rare] = 10  (n_cal = 250)
  clustering executed in 0.0% of replications, 7.89 classes in the null cluster on average
  clustered : WCU 0.235  AvgSize 1.649
  marginal  : WCU 0.236  AvgSize 1.647
  paired difference (clustered - marginal): -0.0006 (MCSE 0.0005, t = -1.22)

E[N_rare] = 20  (n_cal = 500)
  clustering executed in 0.0% of replications, 6.84 classes in the null cluster on average
  clustered : WCU 0.232  AvgSize 1.636
  marginal  : WCU 0.234  AvgSize 1.630
  paired difference (clustered - marginal): -0.0017 (MCSE 0.0005, t = -3.10)

E[N_rare] = 50  (n_cal = 1250)
  clustering executed in 87.3% of replications, 3.96 classes in the null cluster on average
  cluster